# Improved — retrieve-then-rerank with a local LLM

Baseline **+** a local Qwen3-8B that reranks the retrieved codes. The 8B model loads **4-bit** (~6 GB) so it fits a free Colab **T4**.

See `docs/03_improved.md`.

## 1. Setup

Clone the `course` branch and install. Use a **GPU runtime** (Runtime → Change runtime type → T4 GPU).

In [1]:
!git clone https://github.com/AIVIETNAM-AIO-DinhBao/ViClinicalIE_2.git medextract
%cd medextract
!pip -q install -r requirements.txt
!pip -q install -e .

# Fix lỗi transformers -> torchvision mismatch trên Colab.
# Repo này chỉ xử lý text, không cần torchvision/torchaudio/torchtext.
!pip -q uninstall -y torchvision torchaudio torchtext

Cloning into 'medextract'...
remote: Enumerating objects: 119, done.
remote: Counting objects: 100% (119/119), done.
remote: Compressing objects: 100% (98/98), done.
remote: Total 119 (delta 19), reused 119 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (119/119), 994.60 KiB | 3.55 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/medextract
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.2/507.2 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.8/207.8 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:0

## 2. Knowledge bases

The linking step needs the RxNorm + ICD-10 knowledge bases. Their source files are license-gated, so download them yourself (see `INSTALL.md`) and place them in `data/kb/raw/`, then build the indexes.

> If you already have prebuilt `data/kb/processed/*.parquet` + `*.faiss` (e.g. from Google Drive), copy them into `data/kb/processed/` and skip the build cell.

In [2]:
# after placing the raw sources in data/kb/raw/ :
#!python -m medextract.kb.build_rxnorm
#!python -m medextract.kb.build_icd
#!python -m medextract.kb.index --device auto
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p data/kb/processed
!unzip -o "/content/drive/Shareddrives/MY PLACE/kb_processed.zip" -d data/kb/processed
!ls -lh data/kb/processed

# Unzip official test into a temp dir, then flatten all nested *.txt into data/input/.
# This handles zips shaped like 1.txt, input/1.txt, test/input/1.txt, etc.
!rm -rf data/input data/input_raw
!mkdir -p data/input data/input_raw
!unzip -oq "/content/drive/Shareddrives/MY PLACE/input_txt.zip" -d data/input_raw

from pathlib import Path
import shutil

def _num_key(p):
    return (0, int(p.stem)) if p.stem.isdigit() else (1, p.name)

raw = Path('data/input_raw')
dst = Path('data/input')
txts = sorted(raw.rglob('*.txt'), key=_num_key)
print('txt files found inside zip:', len(txts))
print('examples:', [str(p) for p in txts[:10]])
if not txts:
    raise RuntimeError('No *.txt found after unzip. Check Drive path / zip content.')
names = [p.name for p in txts]
dups = sorted({n for n in names if names.count(n) > 1})
if dups:
    raise RuntimeError(f'Duplicate txt names inside zip: {dups[:10]}')
for p in txts:
    shutil.copy2(p, dst / p.name)
direct_txts = sorted(dst.glob('*.txt'), key=_num_key)
print('direct data/input txt:', len(direct_txts))
print('first files:', [p.name for p in direct_txts[:10]])

req = ['ICD10.faiss','ICD10_meta.parquet','icd_terms.parquet','RXNORM.faiss','RXNORM_meta.parquet','rxnorm_terms.parquet']
base = Path('data/kb/processed')
missing = [x for x in req if not (base / x).exists()]
print('missing KB files:', missing)
if missing:
    raise FileNotFoundError(missing)

Mounted at /content/drive
Archive:  /content/drive/Shareddrives/MY PLACE/kb_processed.zip
  inflating: data/kb/processed/ICD10.faiss  
  inflating: data/kb/processed/ICD10_meta.parquet  
  inflating: data/kb/processed/icd_terms.parquet  
  inflating: data/kb/processed/RXNORM.faiss  
  inflating: data/kb/processed/RXNORM_meta.parquet  
  inflating: data/kb/processed/rxnorm_terms.parquet  
total 155M
-rw-r--r-- 1 root root  47M Jul 25 22:49 ICD10.faiss
-rw-r--r-- 1 root root 360K Jul 25 22:49 ICD10_meta.parquet
-rw-r--r-- 1 root root 633K Jul 25 22:39 icd_terms.parquet
-rw-r--r-- 1 root root 106M Jul 25 23:08 RXNORM.faiss
-rw-r--r-- 1 root root 825K Jul 25 23:08 RXNORM_meta.parquet
-rw-r--r-- 1 root root 825K Jul 25 22:37 rxnorm_terms.parquet
Archive:  /content/drive/Shareddrives/MY PLACE/input_txt.zip
   creating: data/input/input/
  inflating: data/input/input/1.txt  
  inflating: data/input/input/10.txt  
  inflating: data/input/input/100.txt  
  inflating: data/input/input/11.txt  
 

## 3. Run the improved pipeline on the sample notes

First run downloads Qwen3-8B (a few minutes). `configs/improved.yaml` sets `load_in_4bit: true`.

In [3]:
!python -c "from pathlib import Path; p=Path('configs/improved.yaml'); s=p.read_text(); s=s.replace('/mnt/pretrained_fm/Qwen_Qwen3-8B', 'Qwen/Qwen3-8B'); p.write_text(s); print(p.read_text())"

# Improved: baseline + a local LLM (Qwen3-8B) that reranks the retrieved
# candidates (retrieve-then-rerank). Retrieval places the correct code in the
# top-k but not always at rank 1; the LLM picks the exact one, constrained to
# the retrieved codes so it can never hallucinate an invalid code.
#
# load_in_4bit fits the 8B model in ~6 GB so it runs on a free Colab T4. For a
# full-precision run on the submission GPU, set load_in_4bit: false and
# device: wait (min_free_gb: 18).
extends: baseline.yaml
solution: improved

llm:
  model: Qwen/Qwen3-8B   # or Qwen/Qwen3-8B (downloads from the Hub)
  device: auto
  dtype: bfloat16
  load_in_4bit: true
  min_free_gb: 6.0
  enable_thinking: false

normalization:
  llm_rerank:
    retrieve_k: 20                # feed the LLM a wider candidate list
    max_candidates:
      ICD10: 2
      RXNORM: 1



In [4]:
!python run.py --config configs/improved.yaml --input data/sample_input --output out/demo_imp

2026-07-26 10:06:49,757 INFO numexpr.utils: NumExpr defaulting to 2 threads.
2026-07-26 10:06:50,657 INFO medextract.llm.engine: loading LLM Qwen/Qwen3-8B on cuda:0 (bfloat16, 4-bit)
2026-07-26 10:06:50,822 INFO httpx: HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-07-26 10:06:50,911 INFO httpx: HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-8B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-26 10:06:50,911 WARNING huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-07-26 10:06:50,922 INFO httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-8B/b968826d9c46dd6066d109eabc6255188de91218/config.json "HTTP/1.1 200 OK"
2026-07-26 10:06:50,935 INFO httpx: HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-8B/b968826d9c46dd6066d109eabc6255188de91218/config.

## 4. Score baseline vs improved on the dev set

In [5]:
!python run.py --config configs/improved.yaml --input data/dev/input --output out/dev_imp
!python score.py --pred out/dev_imp --gold data/dev

2026-07-26 10:14:11,849 INFO numexpr.utils: NumExpr defaulting to 2 threads.
2026-07-26 10:14:12,543 INFO medextract.llm.engine: loading LLM Qwen/Qwen3-8B on cuda:0 (bfloat16, 4-bit)
2026-07-26 10:14:12,716 INFO httpx: HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-8B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-26 10:14:12,717 WARNING huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-07-26 10:14:12,727 INFO httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-8B/b968826d9c46dd6066d109eabc6255188de91218/config.json "HTTP/1.1 200 OK"
2026-07-26 10:14:12,822 INFO httpx: HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-8B/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-26 10:14:12,833 INFO httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-8B/b

## 5. Build a submission zip

Point `--input` at the official test directory; `--zip` writes `out/sub/submission.zip`.

In [6]:
from pathlib import Path

def _num_key(p):
    return (0, int(p.stem)) if p.stem.isdigit() else (1, p.name)

inp = Path('data/input')
txts = sorted(inp.glob('*.txt'), key=_num_key)
nested = sorted(inp.rglob('*.txt'), key=_num_key)
print('direct input txt:', len(txts))
print('nested input txt:', len(nested))
print('first direct files:', [p.name for p in txts[:10]])
assert len(txts) == 100, f'Expected 100 direct txt files in data/input, got {len(txts)}. Nested txt examples: {[str(p) for p in nested[:10]]}'

!rm -rf out/sub
!python run.py --config configs/improved.yaml --input data/input --output out/sub --zip

import json, zipfile
zip_path = Path('out/sub/submission.zip')
jsons = sorted(Path('out/sub').glob('*.json'), key=_num_key)
print('json files written:', len(jsons))
print('zip exists:', zip_path.exists(), 'size:', zip_path.stat().st_size if zip_path.exists() else None)
assert len(jsons) == 100, f'Expected 100 output json files, got {len(jsons)}'
with zipfile.ZipFile(zip_path) as zf:
    names = zf.namelist()
print('zip entries:', len(names), names[:10])
assert len(names) == 100, f'Expected 100 files in zip, got {len(names)}'
assert all('/' not in n and n.endswith('.json') for n in names), names[:10]

2026-07-26 10:18:56,398 INFO numexpr.utils: NumExpr defaulting to 2 threads.
2026-07-26 10:18:56,895 INFO medextract.llm.engine: loading LLM Qwen/Qwen3-8B on cuda:0 (bfloat16, 4-bit)
2026-07-26 10:18:57,062 INFO httpx: HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-8B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-26 10:18:57,074 INFO httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-8B/b968826d9c46dd6066d109eabc6255188de91218/config.json "HTTP/1.1 200 OK"
2026-07-26 10:18:57,167 INFO httpx: HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-8B/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-26 10:18:57,167 WARNING huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-07-26 10:18:57,178 INFO httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-8B/b